In [2]:
import pandas as pd
import json

In [3]:
df=pd.read_csv('llm_input_sample.csv')
df.tail()

,Date,brent_close,wti_close,brent_wti_spread,brent_ret_1d,brent_ret_5d,brent_ret_20d,wti_ret_1d,wti_ret_5d,wti_ret_20d,...,cluster_5,cluster_6,cluster_7,cluster_8,news_impact_brent_day0_value,news_impact_brent_day1_value,news_impact_brent_day2_value,news_impact_brent_day3_value,news_impact_brent_day4_value,news_impact_brent_total
2926,2025-11-17,64.199997,59.910000,4.289997,-0.002951,0.002185,0.052286,-0.002996,-0.003659,0.041551,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2927,2025-11-18,64.889999,60.740002,4.149998,0.010748,-0.004144,0.058219,0.013854,-0.004915,0.050502,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2928,2025-11-19,63.509998,59.439999,4.070000,-0.021267,0.012757,0.014699,-0.021403,0.016242,0.016068,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2929,2025-11-20,63.380001,59.139999,4.240002,-0.002047,0.005872,-0.039551,-0.005047,0.007667,-0.042887,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2930,2025-11-21,62.560001,58.060001,4.500000,-0.012938,-0.028421,-0.051259,-0.018262,-0.033783,-0.055935,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
with open('../data/embedded.json', 'r', encoding='utf-8') as f:
    news = json.load(f)

news[-1]

{'title': 'North Dakota Oil Train Conflagration Prompts Increased Federal Scrutiny',
 'url': 'https://oilprice.com/Energy/Crude-Oil/North-Dakota-Oil-Train-Conflagration-Prompts-Increased-Federal-Scrutiny.html',
 'published': 'Jan 02, 2014 at 13:22',
 'category': '국제 유가 동향',
 'content': 'The year 2013 has ended on a worrying note for oil producers in North America’s Bakken region, as a series of recent train incidents in both Canada and the U.S. have highlighted the hazards of moving crude oil by rail.\n\nOn 30 December at 2:12 p.m. a mile-long train of 106 tankers carrying Bakken crude collided with a derailed 112 car westbound grain train carrying soybeans about a mile west of Casselton, a city of 2,432 people about 20 miles west of Fargo. The crash triggered a series of massive explosions sending toxic fumes into the air. Local authorities said that 18 tanker cars caught fire and the Cass County Sheriff\'s Office “strongly” recommended evacuating Casselton and anyone residing five mi

In [5]:
with open('extra_embedded (1).json', 'r', encoding='utf-8') as f:
    news19 = json.load(f)

news19[-1]

{'title': 'U.S. Gasoline Inventories Sink To 12-Year Lows',
 'url': 'https://oilprice.com/Energy/Crude-Oil/US-Gasoline-Inventories-Sink-To-12-Year-Lows.html',
 'published': 'Nov 19, 2025 at 16:11',
 'category': '국제 유가 동향',
 'content': "Previously, we reported that the pivot by Indian refiners away from Russian oil has triggered a spike in oil product prices even as crude prices remain largely unchanged. To wit, ICE Brent-Gasoil crack spreads doubled from the $15-17/bbl range held in the first half of the year, to a 21-month high above $32/bbl, good for a nearly 70% increase in the year-to-date. Gasoil is a middle distillate mainly used in commercial and agricultural sectors for off-road vehicles, machinery, and generators. And now reports have emerged that the distillates market continues to tighten even as crude prices remain weak. According to new data by the Energy Information Administration (EIA), gasoline inventories clocked in at 205.06 million barrels (mb) for the week ending 7t

In [6]:

import umap
import hdbscan
import numpy as np

embeddings = np.array([item["summary_embedding"] for item in news])

# 1) 64 → 20차원 UMAP
umap_model = umap.UMAP(
    n_components=20,
    n_neighbors=30,   # 조금 더 global 구조 보고
    min_dist=0.0,
    random_state=42
)
emb_20d = umap_model.fit_transform(embeddings)

# 2-a) 20차원에서 HDBSCAN (좀 더 잘게)
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=50,      # 전체 뉴스 개수 보고 조정
    min_samples=10,
    metric='euclidean',
    prediction_data=True 
)
labels_hdb = clusterer.fit_predict(emb_20d)

for item, label in zip(news, labels_hdb):
    item["cluster_hdbscan_20d"] = int(label)

# 2-b) 혹은 20차원에서 KMeans 30개
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=30, random_state=42, n_init=10)
labels_km = kmeans.fit_predict(emb_20d)

for item, label in zip(news, labels_km):
    item["cluster_umap20_kmeans30"] = int(label)

c:\Users\SKAX\Desktop\project\Market-Plan-B-AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\SKAX\Desktop\project\Market-Plan-B-AI\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\SKAX\Desktop\project\Market-Plan-B-AI\venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\SKAX\Desktop\project\Market-Plan-B-AI\venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [7]:
import joblib
joblib.dump(umap_model, "../ai/model_weight/umap_64to20.model")
joblib.dump(kmeans, "../ai/model_weight/kmeans_20d_30clusters.model")
joblib.dump(clusterer, "../ai/model_weight/hdbscan_20d.model")


['../ai/model_weight/hdbscan_20d.model']

In [6]:
print(type(news19))
print(len(news19))
print(news19[0].keys())


<class 'list'>
2
dict_keys(['title', 'url', 'published', 'category', 'content', 'group_size', 'event_uuid', 'summary', 'sentiment', 'trust', 'relation_nation', 'summary_embedding'])


In [7]:
for item in news19:
    emb = np.array(item["summary_embedding"]).reshape(1, -1)
    emb_20d = umap_model.transform(emb)

    # Predict kmeans
    km_label = kmeans.predict(emb_20d)[0]

    # Predict hdbscan
    hdb_label, strength = hdbscan.approximate_predict(clusterer, emb_20d)
    hdb_label, strength = hdb_label[0], strength[0]

    item["cluster_kmeans30"] = int(km_label)
    item["cluster_hdbscan20"] = int(hdb_label)
    item["hdbscan_strength"] = float(strength)


In [8]:
for item in news19:
    print(item["cluster_kmeans30"])

0
0


In [9]:
import json

def build_countermeasure_prompt(df_row_19, news_items, pred_result):
    """
    df_row_19 : 2025-11-19 데이터 1 row (dict 형태)
    news_items: 뉴스 객체 리스트 (2개 이상 가능)
    pred_result: {"pred_return": float, "today_close": float, "predicted_next_close": float}
    """

    # 1) 뉴스 여러 개를 텍스트로 묶기
    news_text = ""
    for idx, item in enumerate(news_items, start=1):
        news_text += f"""
-----------------------------
[뉴스 {idx}]
-----------------------------
제목: {item.get('title')}
요약: {item.get('summary')}
감성 점수: {item.get('sentiment', {}).get('score')}
신뢰도 점수: {item.get('trust', {}).get('score')}
본문 일부:
{item.get("content")[:600]} ...
"""

    # 2) 최종 프롬프트
    prompt = f"""
당신은 원유 시장 분석을 수행하는 전문 애널리스트입니다.
아래 ‘11월 19일 시장 데이터 / 뉴스 2개 / 모델 예측값’을 기반으로
**11월 20일의 시장 대응전략 3가지를** 작성해 주세요.

========================================================
[1] 2025-11-19 원유 시장 정형 데이터
========================================================
{json.dumps(df_row_19, ensure_ascii=False, indent=2)}

========================================================
[2] 2025-11-19 뉴스 데이터 (총 {len(news_items)}개)
========================================================
{news_text}

========================================================
[3] 모델 예측 결과 (BiGRU)
========================================================
예측된 5일 수익률: {pred_result["pred_return"]:.4f}
현재 Brent 종가: {pred_result["today_close"]:.2f} USD
예측된 다음 종가: {pred_result["predicted_next_close"]:.2f} USD

========================================================
[요구사항]
========================================================
- 위 데이터를 종합하여 **시장 대응전략 3개**를 작성해 주세요.
- 각 전략은 아래 형식을 따라야 합니다:

1) 전략명  
2) 적용 기간 (단기 / 중기 / 장기 중 선택)  
3) 전제 조건  
4) 실행 액션 (구체적으로)  
5) 데이터 기반 근거  
   - 정형 데이터(df_row_19)
   - 뉴스 주요 내용(2개)
   - 감성점수 / 신뢰도 점수
   - 모델 예측값(예: 다음날 하락 예상 등)
   각각을 근거로 명확하게 연결해서 설명하세요.

- 반드시 한국어 존댓말로 작성하세요.
- ‘기업 MI 팀에서 바로 사용할 수 있는 수준’으로 구체적으로 작성하세요.
- 투자 조언 형태가 아니라 **리스크 인사이트 + 대응전략**의 형태로 작성하세요.

"""

    return prompt


In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    project=os.getenv("OPENAI_PROJECT_ID")
)


# ============================
# 2) Countermeasure API 함수
# ============================
def generate_countermeasures(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.2,
        messages=[
            {"role": "system", "content": "당신은 원유·정유 시장 전략을 제시하는 에너지 분석 전문가입니다."},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content


# ============================
# 3) 데이터 준비
# ============================

df_row_19 = df.loc[df["Date"] == "2025-11-19"].iloc[0].to_dict()

news_item = news19  # 이미 하나 선택되어 있는 dict

pred_result = {
    "pred_return": -0.012175140902400017,
    "today_close": 64.88999938964844,
    "predicted_next_close": 64.09995450392282
}

# prompt 생성
prompt = build_countermeasure_prompt(df_row_19, news_item, pred_result)

# ============================
# 4) LLM 실행
# ============================
result = generate_countermeasures(prompt)
print(result)


In [11]:
당신은 기업 Market Intelligence(MI) 애널리스트입니다.  
아래 제공되는 자료를 기반으로 **11월 19일자 Daily Oil Market Report**를 작성해 주세요.

자료는 다음 다섯 가지입니다:
1) 정형 데이터 (브렌트/WTI 가격 및 기술지표)
2) 당일 뉴스 목록 (여러 개)
3) 모델 예측값 (익일 Brent 예상수익률 및 예상종가)
4) XAI 결과 (모델이 중요하게 본 상위 변수)
5) 대응 전략 (이미 계산된 2~3개 전략)

---

## 📌 요구사항

### **[1] 전체 구성**
아래 순서대로 리포트를 작성하세요:

### 1) Executive Summary (3~4문장)
- 당일 시장 핵심 인사이트
- 정형 데이터 + 뉴스 + 예측값을 통합한 간단 요약

### 2) Market Data Overview
- 아래 정형 데이터 원문을 표 형태로 요약
- 단기 모멘텀, 변동성, 이동평균 상태를 함께 해석
- Brent/WTI 스프레드의 의미까지 분석

정형데이터:
{정형데이터}

---

### 3) Daily News Analysis
- 주어진 뉴스 목록 모두를 **요약 + 영향 해석**
- 뉴스 기반의 단기/중기 리스크 요인 분리
- 공급, 재고, 지정학, 제품 크랙 등 테마별 분석

뉴스:
{뉴스목록}

---

### 4) Model-driven Outlook (GRU 기반)
- 모델 예측값(수익률, 종가)을 바탕으로 **다음날 방향성** 서술
- 예측값의 크기보다 “해석 중심”으로 작성
- 예측 결과가 뉴스 및 정형 데이터와 어떻게 align되는지 설명

모델 예측:
{모델예측}

---

### 5) XAI Interpretation
- XAI에서 나온 상위 변수들의 의미를 분석
- 해당 변수들이 왜 중요했는지 직관적으로 설명
- 긍정/부정 기여도를 간단한 bullet로 표시

XAI 결과:
{XAI결과}

---

### 6) Recommended Actions (MI 대응 전략)
- 이미 제공된 대응전략 2~3개를 **실제 MI 대응책처럼 정제하여 작성**
- 사내 구매/트레이딩/전략실이 바로 사용할 수 있게 작성

대응전략:
{대응전략}

---

### 7) 결론 요약 (2~3문장)
- 내일 시장에 대한 핵심 포인트
- 주의해야 할 리스크 1개 + 기대 요인 1개를 제시

---

## 📌 작성 스타일
- 존댓말 사용
- 기업 MI 레포트처럼 간결하지만 전문적으로
- 수치는 과도하게 나열하지 말고 “맥락 중심”
- 과도한 투기 조언 금지
- 뉴스–데이터–모델이 어떻게 서로 보완되는지 강조



SyntaxError: unmatched ')' (1198519309.py, line 5)

In [ ]:
df.loc[df["Date"] == "2025-11-19"].iloc[0]

Date                            2025-11-19
brent_close                      63.509998
wti_close                        59.439999
brent_wti_spread                      4.07
brent_ret_1d                     -0.021267
brent_ret_5d                      0.012757
brent_ret_20d                     0.014699
wti_ret_1d                       -0.021403
wti_ret_5d                        0.016242
wti_ret_20d                       0.016068
brent_ma_5                       63.999998
brent_ma_20                        64.4365
brent_ma_60                         65.434
wti_ma_5                            59.774
wti_ma_20                           60.285
wti_ma_60                        61.458167
brent_vol_5d                      0.016156
wti_vol_5d                        0.017203
high_low_range                    0.031334
cluster_0                              NaN
cluster_1                              NaN
cluster_2                              NaN
cluster_3                              NaN
cluster_4  